In [1]:
## #
## # Preparacion de datos para el LAB:
## #
## # https://www.kaggle.com/code/vijayaadithyanvg/car-price-prediction-used-cars
## #
## import pandas as pd  # type : ignore
##
## df = pd.read_csv("../files/input/car_data.csv")
## train = df.sample(frac=0.7, random_state=1)
## test = df.drop(train.index)
## train.to_csv(
##     "../files/input/train_data.csv.zip",
##     index=False,
##     compression="zip",
## )
## test.to_csv(
##     "../files/input/test_data.csv.zip",
##     index=False,
##     compression="zip",
## )

In [2]:
#
# En este dataset se desea pronosticar el precio de vhiculos usados. El dataset
# original contiene las siguientes columnas:
#
# - Car_Name: Nombre del vehiculo.
# - Year: Año de fabricación.
# - Selling_Price: Precio de venta.
# - Present_Price: Precio actual.
# - Driven_Kms: Kilometraje recorrido.
# - Fuel_type: Tipo de combustible.
# - Selling_Type: Tipo de vendedor.
# - Transmission: Tipo de transmisión.
# - Owner: Número de propietarios.
#
# El dataset ya se encuentra dividido en conjuntos de entrenamiento y prueba
# en la carpeta "files/input/".
#
# Los pasos que debe seguir para la construcción de un modelo de
# pronostico están descritos a continuación.
#

import pandas as pd  #  type: ignore

raw_train_df = pd.read_csv(
    "../files/input/train_data.csv.zip",
    compression="zip",
)

raw_test_df = pd.read_csv(
    "../files/input/test_data.csv.zip",
    compression="zip",
)

display(raw_train_df.head())
display(raw_test_df.head())

,Car_Name,Year,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner
0,jazz,2016,7.40,8.500,15059,Petrol,Dealer,Automatic,0
1,i10,2013,4.00,4.600,30000,Petrol,Dealer,Manual,0
2,TVS Apache RTR 180,2011,0.50,0.826,6000,Petrol,Individual,Manual,0
3,eon,2016,3.15,4.430,15000,Petrol,Dealer,Manual,0
4,Royal Enfield Thunder 350,2013,1.25,1.500,15000,Petrol,Individual,Manual,0


,Car_Name,Year,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner
0,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
1,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
2,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
3,ciaz,2015,6.75,8.12,18796,Petrol,Dealer,Manual,0
4,s cross,2015,6.50,8.61,33429,Diesel,Dealer,Manual,0


In [3]:
## from pprint import pprint
##
## pprint(sorted(raw_train_df.Year.value_counts().index.to_list()))
## display(raw_train_df.Year.isna().sum())
## pprint(sorted(raw_test_df.Year.value_counts().index.to_list()))
## display(raw_test_df.Year.isna().sum())

In [4]:
#
# Paso 1.
# Preprocese los datos.
# - Cree la columna 'Age' a partir de la columna 'Year'.
#   Asuma que el año actual es 2021.
# - Elimine las columnas 'Year' y 'Car_Name'.
#
def preprocess(df):
    df["Age"] = 2021 - df["Year"]
    df.drop(columns=["Year", "Car_Name"], inplace=True)
    return df


clean_train_df = preprocess(raw_train_df.copy())
clean_test_df = preprocess(raw_test_df.copy())

In [5]:
clean_train_df.head()

,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner,Age
0,7.40,8.500,15059,Petrol,Dealer,Automatic,0,5
1,4.00,4.600,30000,Petrol,Dealer,Manual,0,8
2,0.50,0.826,6000,Petrol,Individual,Manual,0,10
3,3.15,4.430,15000,Petrol,Dealer,Manual,0,5
4,1.25,1.500,15000,Petrol,Individual,Manual,0,8


In [6]:
clean_test_df.head()

,Selling_Price,Present_Price,Driven_kms,Fuel_Type,Selling_type,Transmission,Owner,Age
0,4.75,9.54,43000,Diesel,Dealer,Manual,0,8
1,7.25,9.85,6900,Petrol,Dealer,Manual,0,4
2,2.85,4.15,5200,Petrol,Dealer,Manual,0,10
3,6.75,8.12,18796,Petrol,Dealer,Manual,0,6
4,6.50,8.61,33429,Diesel,Dealer,Manual,0,6


In [12]:
#
# Paso 2.
# Divida los datasets en x_train, y_train, x_test, y_test.
#
import os
import pickle

x_train = clean_train_df.drop(columns=["Present_Price"])
y_train = clean_train_df["Present_Price"]

x_test = clean_test_df.drop(columns=["Present_Price"])
y_test = clean_test_df["Present_Price"]


if not os.path.exists("../files/grading/"):
    os.makedirs("../files/grading/")

with open("../files/grading/x_train.pkl", "wb") as f:
    pickle.dump(x_train, f)

with open("../files/grading/y_train.pkl", "wb") as f:
    pickle.dump(y_train, f)

with open("../files/grading/x_test.pkl", "wb") as f:
    pickle.dump(x_test, f)

with open("../files/grading/y_test.pkl", "wb") as f:
    pickle.dump(y_test, f)

In [ ]:
#
# Paso 3.
# Cree un pipeline para el modelo de clasificación. Este pipeline debe
# contener las siguientes capas:
# - Transforma las variables categoricas usando el método
#   one-hot-encoding.
# - Escala las variables numéricas al intervalo [0, 1].
# - Selecciona las K mejores entradas.
# - Ajusta un modelo de regresion lineal.
#
from sklearn.compose import ColumnTransformer  # type: ignore
from sklearn.linear_model import LinearRegression  # type: ignore
from sklearn.pipeline import Pipeline  # type: ignore
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler  # type: ignore
from sklearn.feature_selection import SelectKBest, f_regression  # type: ignore


pipeline = Pipeline(
    [
        (
            "transformer",
            ColumnTransformer(
                [
                    (
                        "encoder",
                        OneHotEncoder(),
                        ["Fuel_Type", "Selling_type", "Transmission"],
                    ),
                ],
                remainder=MinMaxScaler(),
            ),
        ),
        ("selectkbest", SelectKBest(f_regression)),
        ("model", LinearRegression()),
    ]
)

In [ ]:
#
# Paso 4.
# Optimice los hiperparametros del pipeline usando validación cruzada.
# Use 10 splits para la validación cruzada. Use el error medio absoluto
# para medir el desempeño modelo.
#
import warnings
from sklearn.model_selection import GridSearchCV  # type: ignore

warnings.filterwarnings("ignore")

param_grid = {
    "selectkbest__k": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
}

grid_search_pipeline = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=10,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)

grid_search_pipeline.fit(x_train, y_train)

print(grid_search_pipeline.best_estimator_)
print(grid_search_pipeline.score(x_train, y_train))
print(grid_search_pipeline.score(x_test, y_test))

Pipeline(steps=[('transformer',
                 ColumnTransformer(remainder=MinMaxScaler(),
                                   transformers=[('encoder', OneHotEncoder(),
                                                  ['Fuel_Type', 'Selling_type',
                                                   'Transmission'])])),
                ('selectkbest',
                 SelectKBest(k=11,
                             score_func=<function f_regression at 0x125da2550>)),
                ('model', LinearRegression())])
-1.5999810426540284
-2.4292222222222226


In [10]:
#
# Paso 5.
# Salve el modelo como "files/models/model.pkl".
#
import os
import pickle

if not os.path.exists("../files/models"):
    os.makedirs("../files/models")

with open("../files/models/model.pkl", "wb") as file:
    pickle.dump(grid_search_pipeline, file)

In [11]:
#
# Paso 6.
# Calcule las metricas r2, error cuadratico medio, y error absoluto medio
# para los conjuntos de entrenamiento y prueba. Guardelas en el archivo
# files/output/metrics.json. Cada fila del archivo es un diccionario con
# las metricas de un modelo. Este diccionario tiene un campo para indicar
# si es el conjunto de entrenamiento o prueba. Por ejemplo:
#
# {'type': 'metrics', 'dataset': 'train', 'r2': 0.8, 'mse': 0.7, 'mad': 0.9}
# {'type': 'metrics', 'dataset': 'test', 'r2': 0.7, 'mse': 0.6, 'mad': 0.8}
#
import os

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error  # type: ignore

if not os.path.exists("../files/output"):
    os.makedirs("../files/output")

metrics = {
    "type": "metrics",
    "dataset": "train",
    "r2": float(r2_score(y_train, grid_search_pipeline.predict(x_train))),
    "mse": float(mean_squared_error(y_train, grid_search_pipeline.predict(x_train))),
    "mad": float(mean_absolute_error(y_train, grid_search_pipeline.predict(x_train))),
}

with open("../files/output/metrics.json", "w") as file:
    file.write(str(metrics).replace("'", '"'))
    file.write("\n")

display(metrics)

metrics = {
    "type": "metrics",
    "dataset": "test",
    "r2": float(r2_score(y_test, grid_search_pipeline.predict(x_test))),
    "mse": float(mean_squared_error(y_test, grid_search_pipeline.predict(x_test))),
    "mad": float(mean_absolute_error(y_test, grid_search_pipeline.predict(x_test))),
}

with open("../files/output/metrics.json", "a") as file:
    file.write(str(metrics).replace("'", '"'))
    file.write("\n")

display(metrics)

{'type': 'metrics',
 'dataset': 'train',
 'r2': 0.8903242447560339,
 'mse': 5.949066246445498,
 'mad': 1.5999810426540284}

{'type': 'metrics',
 'dataset': 'test',
 'r2': 0.7297628240859133,
 'mse': 32.90872680555555,
 'mad': 2.4292222222222226}